# RAG Pipeline — Build & Deploy

This notebook handles the **setup, compilation, and deployment** of the RAG pipeline on Red Hat OpenShift AI (RHOAI).

## Pipeline Overview

The pipeline has **up to 5 steps**, orchestrated by Kubeflow Pipelines (KFP):

| Step | Component | Description |
|------|-----------|-------------|
| 1 | **Parse & Chunk** | RayJob distributes PDF parsing via Docling + HybridChunker, writes JSONL to S3 (MinIO) |
| 2 | **Deploy Embedding** *(optional)* | Deploys embedding model as a KServe InferenceService (only when `deploy_embedding=True`) |
| 3 | **Ingest to Milvus** | Reads chunks from S3, embeds with local or deployed model, inserts into Milvus |
| 4 | **Download Model** | Downloads LLM weights to a PVC (cached — skips if already present) |
| 5 | **Deploy Model** | Creates a vLLM InferenceService from the cached PVC model |

Steps 1→3 form the **data chain** and steps 4→5 form the **model chain** — both chains run in parallel.

## Embedding Modes

This pipeline supports **two modes** for generating embeddings:

**Mode 1: Local** (default)
- Embedding model runs inside the pipeline component
- Uses `sentence-transformers` library
- Simpler setup, good for testing

**Mode 2: Service**
- Embedding model deployed as a dedicated InferenceService
- Reusable across multiple runs and queries
- Better resource utilization, can leverage GPU
- Ideal for production

Toggle between modes by setting `EMBEDDING_MODE = "local"` or `"service"` in the configuration.

## What This Notebook Covers

1. **Prerequisites** — Create PVCs, Secrets, RBAC
2. **Deploy Embedding Service** (optional - for service mode)
3. **Compile the Pipeline** — Generate the KFP YAML
4. **Upload & Run** — Submit to Data Science Pipelines
5. **Monitor** — Track pipeline execution

---
## 1. Prerequisites

Before running the pipeline, ensure the following are in place:

- **RHOAI** installed with Data Science Pipelines (DSPA) configured
- **KubeRay** operator enabled
- **Milvus** deployed (e.g., via Milvus Operator)
- **MinIO** or S3-compatible storage for intermediate chunks
- **GPU nodes** with NVIDIA GPUs for model serving

### OpenShift Login

If running **outside** the cluster (e.g., local laptop), log in first.
If running **inside** the cluster (RHOAI notebook server), skip this cell — the service account token is used automatically.

In [ ]:
import subprocess

# Check if already logged in
result = subprocess.run(["oc", "whoami"], capture_output=True, text=True)
if result.returncode == 0:
    print(f"✅ Logged in as: {result.stdout.strip()}")
    
    # Show current cluster
    cluster = subprocess.run(["oc", "whoami", "--show-server"], capture_output=True, text=True)
    print(f"   Cluster: {cluster.stdout.strip()}")
else:
    print("❌ Not logged in to OpenShift.")
    print("   Run the following command first:")
    print("   oc login --token=<your-token> --server=<your-cluster-url>")

In [ ]:
# Install required packages
!pip install -q kfp kfp-kubernetes kubernetes codeflare-sdk pyyaml tabulate requests

In [ ]:
# ============================================================
# Configuration — Update these values for your environment
# ============================================================

NAMESPACE = "ray-docling"

# S3 / MinIO
S3_ENDPOINT = "http://minio-service.default.svc.cluster.local:9000"
S3_BUCKET = "rag-chunks"
S3_PREFIX = "chunks"
S3_SECRET_NAME = "minio-secret"  # K8s Secret with keys: access_key, secret_key

# PDF Input
INPUT_PATH = "input/pdfs"  # Path relative to pvc_mount_path

# Milvus
MILVUS_HOST = "milvus-milvus.milvus.svc.cluster.local"
MILVUS_PORT = 19530
COLLECTION_NAME = "paper_rag_documents"

# Embedding Configuration
EMBEDDING_MODE = "local"  # Options: "local" or "service"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_DIM = 384
# Only used when EMBEDDING_MODE = "service"
EMBEDDING_SERVICE_NAME = "embedding-service"
EMBEDDING_ENDPOINT = f"http://{EMBEDDING_SERVICE_NAME}.{NAMESPACE}.svc.cluster.local:8080"

# LLM
LLM_MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
LLM_SERVED_NAME = "mistral-7b-instruct-v0-3"
MODEL_CACHE_PVC = "model-cache-pvc"
INFERENCE_URL = "http://mistral-7b-instruct-internal.ray-docling.svc.cluster.local:8080/v1"

print(f"Namespace:      {NAMESPACE}")
print(f"S3 endpoint:    {S3_ENDPOINT}")
print(f"Milvus:         {MILVUS_HOST}:{MILVUS_PORT}")
print(f"Collection:     {COLLECTION_NAME}")
print(f"Input path:     {INPUT_PATH}")
print(f"Embedding mode: {EMBEDDING_MODE}")
if EMBEDDING_MODE == "service":
    print(f"Embedding URL:  {EMBEDDING_ENDPOINT}")
else:
    print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"LLM endpoint:   {INFERENCE_URL}")

### Initialize Kubernetes Client

In [ ]:
from kubernetes import client, config
from kubernetes.client.rest import ApiException
import yaml

# Load kubeconfig
try:
    config.load_incluster_config()
except:
    config.load_kube_config()

v1 = client.CoreV1Api()
rbac_v1 = client.RbacAuthorizationV1Api()
custom_api = client.CustomObjectsApi()

print("✅ Kubernetes client initialized")

### 1.1 Create LLM Internal Service

The InferenceService predictor is headless by default. Create a ClusterIP service for in-cluster access.

In [ ]:
service_name = "mistral-7b-instruct-internal"

try:
    service = v1.read_namespaced_service(name=service_name, namespace=NAMESPACE)
    print(f"✅ Service '{service_name}' exists")
    print(f"   ClusterIP: {service.spec.cluster_ip}")
except ApiException as e:
    if e.status == 404:
        print(f"⚠ Service '{service_name}' not found. Creating it...")
        
        try:
            with open("../manifests/llm-inference-service.yaml", "r") as f:
                service_manifest = yaml.safe_load(f)
            v1.create_namespaced_service(namespace=NAMESPACE, body=service_manifest)
            print(f"✅ Service '{service_name}' created successfully")
        except Exception as create_error:
            print(f"❌ Failed to create service: {create_error}")
            print("You can create it manually:")
            print(f"  kubectl apply -f ../manifests/llm-inference-service.yaml")
    else:
        print(f"❌ Error checking service: {e}")

### 1.2 Create S3 Secret

In [ ]:
secret = client.V1Secret(
    metadata=client.V1ObjectMeta(name=S3_SECRET_NAME, namespace=NAMESPACE),
    string_data={
        "access_key": "minioadmin",
        "secret_key": "minioadmin"
    },
    type="Opaque"
)

try:
    v1.create_namespaced_secret(namespace=NAMESPACE, body=secret)
    print(f"✅ Secret '{S3_SECRET_NAME}' created")
except ApiException as e:
    if e.status == 409:
        v1.patch_namespaced_secret(name=S3_SECRET_NAME, namespace=NAMESPACE, body=secret)
        print(f"✅ Secret '{S3_SECRET_NAME}' updated")
    else:
        print(f"❌ Error creating secret: {e}")

### 1.3 Create Model Cache PVC

In [ ]:
with open("manifests/model-cache-pvc.yaml", "r") as f:
    pvc_manifest = yaml.safe_load(f)

try:
    v1.create_namespaced_persistent_volume_claim(namespace=NAMESPACE, body=pvc_manifest)
    print(f"✅ PVC '{pvc_manifest['metadata']['name']}' created")
except ApiException as e:
    if e.status == 409:
        print(f"✅ PVC '{pvc_manifest['metadata']['name']}' already exists")
    else:
        print(f"❌ Error creating PVC: {e}")

### 1.4 Create RBAC

In [ ]:
with open("manifests/rbac.yaml", "r") as f:
    rbac_docs = yaml.safe_load_all(f)
    
    for doc in rbac_docs:
        if not doc:
            continue
            
        kind = doc.get("kind")
        
        try:
            if kind == "Role":
                rbac_v1.create_namespaced_role(namespace=NAMESPACE, body=doc)
                print(f"✅ Role '{doc['metadata']['name']}' created")
            elif kind == "RoleBinding":
                rbac_v1.create_namespaced_role_binding(namespace=NAMESPACE, body=doc)
                print(f"✅ RoleBinding '{doc['metadata']['name']}' created")
        except ApiException as e:
            if e.status == 409:
                print(f"✅ {kind} '{doc['metadata']['name']}' already exists")
            else:
                print(f"❌ Error creating {kind}: {e}")

### 1.5 Verify Prerequisites

In [ ]:
from tabulate import tabulate

print("=== PVCs ===")
pvcs = v1.list_namespaced_persistent_volume_claim(namespace=NAMESPACE)
pvc_data = [[pvc.metadata.name, pvc.status.phase, pvc.spec.resources.requests.get('storage')] 
            for pvc in pvcs.items]
print(tabulate(pvc_data, headers=["NAME", "STATUS", "CAPACITY"], tablefmt="simple"))

print("\n=== Secrets ===")
try:
    secret = v1.read_namespaced_secret(name=S3_SECRET_NAME, namespace=NAMESPACE)
    print(f"✅ {S3_SECRET_NAME} exists with {len(secret.data)} keys")
except ApiException as e:
    print(f"❌ Secret {S3_SECRET_NAME} not found: {e}")

print("\n=== RBAC ===")
try:
    role = rbac_v1.read_namespaced_role(name="pipeline-rag-role", namespace=NAMESPACE)
    print(f"✅ Role 'pipeline-rag-role' exists with {len(role.rules)} rules")
except ApiException as e:
    print(f"❌ Role not found: {e}")

---
## 2. Deploy Embedding Service (Optional - Service Mode Only)

If `EMBEDDING_MODE = "service"`, deploy the embedding model as an InferenceService.

**Skip this section if using `EMBEDDING_MODE = "local"`**

In [ ]:
if EMBEDDING_MODE == "service":
    # Create ServingRuntime for text-embeddings-inference (TEI)
    serving_runtime = {
        "apiVersion": "serving.kserve.io/v1alpha1",
        "kind": "ServingRuntime",
        "metadata": {
            "name": EMBEDDING_SERVICE_NAME,
            "namespace": NAMESPACE
        },
        "spec": {
            "containers": [{
                "name": "kserve-container",
                "image": "ghcr.io/huggingface/text-embeddings-inference:latest",
                "args": [
                    "--model-id", EMBEDDING_MODEL,
                    "--port", "8080",
                    "--max-batch-tokens", "8192"
                ],
                "ports": [{"containerPort": 8080, "protocol": "TCP"}],
                "resources": {
                    "requests": {"cpu": "2", "memory": "4Gi"},
                    "limits": {"cpu": "4", "memory": "8Gi"}
                }
            }],
            "supportedModelFormats": [{"name": "huggingface", "version": "1"}]
        }
    }
    
    # Create InferenceService
    inference_service = {
        "apiVersion": "serving.kserve.io/v1beta1",
        "kind": "InferenceService",
        "metadata": {
            "name": EMBEDDING_SERVICE_NAME,
            "namespace": NAMESPACE,
            "annotations": {"serving.kserve.io/deploymentMode": "RawDeployment"}
        },
        "spec": {
            "predictor": {
                "model": {
                    "modelFormat": {"name": "huggingface"},
                    "runtime": EMBEDDING_SERVICE_NAME,
                    "resources": {
                        "requests": {"cpu": "2", "memory": "4Gi"},
                        "limits": {"cpu": "4", "memory": "8Gi"}
                    }
                }
            }
        }
    }
    
    # Apply ServingRuntime
    try:
        custom_api.create_namespaced_custom_object(
            group="serving.kserve.io", version="v1alpha1",
            namespace=NAMESPACE, plural="servingruntimes", body=serving_runtime
        )
        print(f"✅ ServingRuntime '{EMBEDDING_SERVICE_NAME}' created")
    except ApiException as e:
        if e.status == 409:
            print(f"✅ ServingRuntime '{EMBEDDING_SERVICE_NAME}' already exists")
        else:
            print(f"❌ Error creating ServingRuntime: {e}")
    
    # Apply InferenceService
    try:
        custom_api.create_namespaced_custom_object(
            group="serving.kserve.io", version="v1beta1",
            namespace=NAMESPACE, plural="inferenceservices", body=inference_service
        )
        print(f"✅ InferenceService '{EMBEDDING_SERVICE_NAME}' created")
        print(f"   Endpoint: {EMBEDDING_ENDPOINT}")
    except ApiException as e:
        if e.status == 409:
            print(f"✅ InferenceService '{EMBEDDING_SERVICE_NAME}' already exists")
        else:
            print(f"❌ Error creating InferenceService: {e}")
else:
    print(f"Skipping embedding service deployment (EMBEDDING_MODE = '{EMBEDDING_MODE}')")

### Check Embedding Service Status

In [ ]:
if EMBEDDING_MODE == "service":
    import time
    from tabulate import tabulate
    
    print("Checking embedding service status...\n")
    
    try:
        isvc = custom_api.get_namespaced_custom_object(
            group="serving.kserve.io", version="v1beta1",
            namespace=NAMESPACE, plural="inferenceservices",
            name=EMBEDDING_SERVICE_NAME
        )
        
        conditions = isvc.get('status', {}).get('conditions', [])
        for cond in conditions:
            print(f"  {cond['type']}: {cond['status']}")
    except ApiException as e:
        print(f"❌ Error: {e}")
    
    print("\n=== Pods ===")
    pods = v1.list_namespaced_pod(
        namespace=NAMESPACE,
        label_selector=f"serving.kserve.io/inferenceservice={EMBEDDING_SERVICE_NAME}"
    )
    
    if pods.items:
        pod_data = []
        for pod in pods.items:
            ready = sum(1 for c in pod.status.container_statuses if c.ready) if pod.status.container_statuses else 0
            total = len(pod.status.container_statuses) if pod.status.container_statuses else 0
            pod_data.append([pod.metadata.name, pod.status.phase, f"{ready}/{total}"])
        print(tabulate(pod_data, headers=["NAME", "STATUS", "READY"], tablefmt="simple"))
    else:
        print("⏳ No pods found yet")
else:
    print(f"Skipping (EMBEDDING_MODE = '{EMBEDDING_MODE}')")

### Test Embedding Service

In [ ]:
if EMBEDDING_MODE == "service":
    import requests
    import time
    
    print(f"Testing embedding endpoint: {EMBEDDING_ENDPOINT}\n")
    
    max_retries = 30
    retry_interval = 5
    
    for i in range(max_retries):
        try:
            resp = requests.post(
                f"{EMBEDDING_ENDPOINT}/v1/embeddings",
                json={"model": EMBEDDING_MODEL, "input": ["test sentence"]},
                timeout=10
            )
            resp.raise_for_status()
            embedding = resp.json()['data'][0]['embedding']
            print(f"✅ Embedding service is ready!")
            print(f"   Embedding dimension: {len(embedding)}")
            break
        except requests.ConnectionError:
            if i < max_retries - 1:
                print(f"⏳ Retrying in {retry_interval}s... ({i+1}/{max_retries})")
                time.sleep(retry_interval)
            else:
                print(f"❌ Service not ready after {max_retries * retry_interval}s")
        except Exception as e:
            print(f"❌ Error: {e}")
            break
else:
    print(f"Skipping (EMBEDDING_MODE = '{EMBEDDING_MODE}')")

---
## 3. Compile the Pipeline

The pipeline is defined in `pipeline_multistep.py`. Compile it to generate the KFP YAML.

In [ ]:
!python3 pipeline_multistep.py

In [ ]:
import os
yaml_path = "rag_multistep_pipeline.yaml"
size_kb = os.path.getsize(yaml_path) / 1024
print(f"✅ Compiled: {yaml_path} ({size_kb:.1f} KB)")

---
## 4. Upload & Run the Pipeline

In [ ]:
import kfp
import subprocess

# Get DSPA route
try:
    route = custom_api.get_namespaced_custom_object(
        group="route.openshift.io", version="v1",
        namespace=NAMESPACE, plural="routes", name="ds-pipeline-dspa"
    )
    dspa_url = f"https://{route['spec']['host']}"
    print(f"DSPA URL: {dspa_url}")
except Exception as e:
    print(f"Error getting route: {e}")
    dspa_route = !oc get route ds-pipeline-dspa -n {NAMESPACE} -o jsonpath='{{.spec.host}}'
    dspa_url = f"https://{dspa_route[0]}"

# Get auth token
token = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True).stdout.strip()

# Create KFP client
kfp_client = kfp.Client(host=dspa_url, existing_token=token, verify_ssl=False)

# Upload pipeline
pipeline = kfp_client.upload_pipeline(
    pipeline_package_path="rag_multistep_pipeline.yaml",
    pipeline_name="rag-multi-step-pipeline"
)
print(f"✅ Pipeline uploaded: {pipeline.pipeline_id}")

### Create Pipeline Run

In [ ]:
# Determine embedding endpoint and deploy flag
embedding_endpoint_param = EMBEDDING_ENDPOINT if EMBEDDING_MODE == "service" else ""
deploy_embedding_param = EMBEDDING_MODE == "service"

# Create run
run = kfp_client.create_run_from_pipeline_package(
    pipeline_file="rag_multistep_pipeline.yaml",
    run_name="rag-test-run",
    arguments={
        "pvc_name": "data-pvc",
        "pvc_mount_path": "/mnt/data",
        "namespace": NAMESPACE,
        "s3_endpoint": S3_ENDPOINT,
        "s3_bucket": S3_BUCKET,
        "s3_prefix": S3_PREFIX,
        "s3_secret_name": S3_SECRET_NAME,
        "input_path": INPUT_PATH,
        "ray_image": "quay.io/rhoai-szaher/docling-ray:latest",
        "num_workers": 2,
        "worker_cpus": 8,
        "worker_memory_gb": 16,
        "cpus_per_actor": 4,
        "min_actors": 2,
        "max_actors": 4,
        "chunk_max_tokens": 256,
        "num_files": 1000,
        "deploy_embedding": deploy_embedding_param,
        "embedding_endpoint": embedding_endpoint_param,
        "embedding_model": EMBEDDING_MODEL,
        "embedding_dim": EMBEDDING_DIM,
        "milvus_host": MILVUS_HOST,
        "milvus_port": MILVUS_PORT,
        "collection_name": COLLECTION_NAME,
        "llm_model_name": LLM_MODEL_NAME,
        "model_cache_pvc": MODEL_CACHE_PVC,
        "max_model_len": 4096,
        "gpu_count": 1,
    },
)

print(f"✅ Run started: {run.run_id}")
print(f"\nConfiguration:")
print(f"  PDFs from: /mnt/data/{INPUT_PATH}")
print(f"  Embedding mode: {EMBEDDING_MODE}")
print(f"  Deploy embedding: {deploy_embedding_param}")
if EMBEDDING_MODE == "service":
    print(f"  Embedding endpoint: {embedding_endpoint_param}")
else:
    print(f"  Embedding model: {EMBEDDING_MODEL} (local)")

---
## 5. Monitor Pipeline Execution

### Step 1: Parse & Chunk (RayJob)

In [ ]:
from tabulate import tabulate

print("=== RayJobs ===")
try:
    rayjobs = custom_api.list_namespaced_custom_object(
        group="ray.io", version="v1", namespace=NAMESPACE, plural="rayjobs"
    )
    
    if rayjobs['items']:
        job_data = []
        for job in rayjobs['items']:
            status = job.get('status', {})
            job_data.append([
                job['metadata']['name'],
                status.get('jobStatus', 'Unknown'),
                status.get('jobDeploymentStatus', 'Unknown')
            ])
        print(tabulate(job_data, headers=["NAME", "JOB STATUS", "DEPLOYMENT STATUS"], tablefmt="simple"))
    else:
        print("No RayJobs found")
except Exception as e:
    print(f"Error: {e}")

print("\n=== Ray Pods ===")
pods = v1.list_namespaced_pod(namespace=NAMESPACE, label_selector="ray.io/node-type")
if pods.items:
    pod_data = [[p.metadata.name, p.status.phase, p.metadata.labels.get('ray.io/node-type')] for p in pods.items]
    print(tabulate(pod_data, headers=["NAME", "STATUS", "NODE TYPE"], tablefmt="simple"))
else:
    print("No Ray pods found")

### Step 2: Ingest to Milvus

In [ ]:
pods = v1.list_namespaced_pod(
    namespace=NAMESPACE,
    label_selector="pipelines.kubeflow.org/v2_component=true"
)

if pods.items:
    sorted_pods = sorted(pods.items, key=lambda p: p.metadata.creation_timestamp)
    pod_data = []
    for pod in sorted_pods:
        created = pod.metadata.creation_timestamp.strftime("%Y-%m-%d %H:%M:%S")
        pod_data.append([pod.metadata.name, pod.status.phase, created])
    print(tabulate(pod_data, headers=["NAME", "STATUS", "CREATED"], tablefmt="simple"))
else:
    print("No pipeline component pods found")

### Steps 3 & 4: Model Download & Deployment

In [ ]:
print("=== InferenceServices ===")
try:
    isvc_list = custom_api.list_namespaced_custom_object(
        group="serving.kserve.io", version="v1beta1",
        namespace=NAMESPACE, plural="inferenceservices"
    )
    
    if isvc_list['items']:
        isvc_data = []
        for isvc in isvc_list['items']:
            status = isvc.get('status', {})
            ready = status.get('conditions', [{}])[0].get('status', 'Unknown') if status.get('conditions') else 'Unknown'
            isvc_data.append([isvc['metadata']['name'], ready, status.get('url', 'N/A')])
        print(tabulate(isvc_data, headers=["NAME", "READY", "URL"], tablefmt="simple"))
    else:
        print("No InferenceServices found")
except Exception as e:
    print(f"Error: {e}")

print("\n=== LLM Serving Pod ===")
pods = v1.list_namespaced_pod(
    namespace=NAMESPACE,
    label_selector=f"serving.kserve.io/inferenceservice={LLM_SERVED_NAME}"
)

if pods.items:
    pod_data = []
    for pod in pods.items:
        ready = sum(1 for c in pod.status.container_statuses if c.ready) if pod.status.container_statuses else 0
        total = len(pod.status.container_statuses) if pod.status.container_statuses else 0
        pod_data.append([pod.metadata.name, pod.status.phase, f"{ready}/{total}"])
    print(tabulate(pod_data, headers=["NAME", "STATUS", "READY"], tablefmt="simple"))
else:
    print(f"No pods found for '{LLM_SERVED_NAME}'")

---
## Next Steps

Once the pipeline completes successfully, use the **`rag_query_test.ipynb`** notebook to:
- Verify the deployed services
- Test RAG queries
- Explore the RAG system interactively

---
## 6. Cleanup (Optional)

Uncomment the sections below to tear down deployed resources.

In [ ]:
# Uncomment the sections you want to execute:

# --- Delete LLM InferenceService ---
# try:
#     custom_api.delete_namespaced_custom_object(
#         group="serving.kserve.io", version="v1beta1",
#         namespace=NAMESPACE, plural="inferenceservices", name=LLM_SERVED_NAME
#     )
#     print(f"Deleted InferenceService '{LLM_SERVED_NAME}'")
# except ApiException as e:
#     print(f"Error: {e}")

# --- Delete LLM ServingRuntime ---
# try:
#     custom_api.delete_namespaced_custom_object(
#         group="serving.kserve.io", version="v1alpha1",
#         namespace=NAMESPACE, plural="servingruntimes", name=LLM_SERVED_NAME
#     )
#     print(f"Deleted ServingRuntime '{LLM_SERVED_NAME}'")
# except ApiException as e:
#     print(f"Error: {e}")

# --- Delete Embedding InferenceService (service mode) ---
# try:
#     custom_api.delete_namespaced_custom_object(
#         group="serving.kserve.io", version="v1beta1",
#         namespace=NAMESPACE, plural="inferenceservices", name=EMBEDDING_SERVICE_NAME
#     )
#     print(f"Deleted InferenceService '{EMBEDDING_SERVICE_NAME}'")
# except ApiException as e:
#     print(f"Error: {e}")
#
# try:
#     custom_api.delete_namespaced_custom_object(
#         group="serving.kserve.io", version="v1alpha1",
#         namespace=NAMESPACE, plural="servingruntimes", name=EMBEDDING_SERVICE_NAME
#     )
#     print(f"Deleted ServingRuntime '{EMBEDDING_SERVICE_NAME}'")
# except ApiException as e:
#     print(f"Error: {e}")

# --- Delete Model Cache PVC ---
# try:
#     v1.delete_namespaced_persistent_volume_claim(name=MODEL_CACHE_PVC, namespace=NAMESPACE)
#     print(f"Deleted PVC '{MODEL_CACHE_PVC}'")
# except ApiException as e:
#     print(f"Error: {e}")

# --- Delete RBAC ---
# try:
#     rbac_v1.delete_namespaced_role(name="pipeline-rag-role", namespace=NAMESPACE)
#     rbac_v1.delete_namespaced_role_binding(name="pipeline-rag-rolebinding", namespace=NAMESPACE)
#     print("Deleted RBAC resources")
# except ApiException as e:
#     print(f"Error: {e}")

# --- Delete S3 Secret ---
# try:
#     v1.delete_namespaced_secret(name=S3_SECRET_NAME, namespace=NAMESPACE)
#     print(f"Deleted Secret '{S3_SECRET_NAME}'")
# except ApiException as e:
#     print(f"Error: {e}")

print("Uncomment the sections above to execute cleanup")